In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find MedGemma-27b-text-it project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import os

os.environ["HF_HOME"] = "/orcd/compute/mghassem/001/gobi1/huggingface"
os.environ["TRANSFORMERS_CACHE"] = "/orcd/compute/mghassem/001/gobi1/huggingface"

# Full path to the model snapshot
model_path = "/orcd/compute/mghassem/001/gobi1/huggingface/hub/models--google--medgemma-27b-text-it/snapshots/5b667cf2ddcf064085bc90952edb35a0edbfb79c"

tokenizer = AutoTokenizer.from_pretrained(
    model_path,
    use_fast=True,
    local_files_only=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    local_files_only=True
)

prompt = "Give me a short introduction to large language model."

messages = [
    {"role": "user", "content": prompt}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)

generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
print(response)

/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|█████████████████████████████████████████| 11/11 [00:07<00:00,  1.42it/s]
The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:1 for open-end generation.


Okay, here's a short introduction to Large Language Models (LLMs):

**Large Language Models (LLMs) are a type of artificial intelligence (AI) designed to understand, generate, and interact with human language.**

Think of them as incredibly sophisticated pattern-matching machines. They are trained on massive amounts of text data (like books, articles, websites) and learn the statistical relationships between words and concepts.

**Key characteristics:**

*   **"Large":** They have billions (or even trillions) of parameters, which are essentially the variables the model adjusts during training to learn patterns.
*   **"Language":** Their primary function is processing and generating human language.
*   **Capabilities:** They can perform tasks like:
    *   Answering questions
    *   Writing essays, code, or creative content
    *   Translating languages
    *   Summarizing text
    *   Holding conversations (like chatbots)

**In essence, LLMs are powerful tools that can mimic human-lik

In [2]:
import pandas as pd 
import re
import torch
import os

# Load data
df = pd.read_csv(paths.DATA / "gpt5-irr-removed-relevancy-combined-dec-12.csv")
print("Columns in dataset:")
print(df.columns.tolist())

# Function to extract answer letter using multiple patterns
def extract_answer_letter(text):
    if pd.isna(text) or not text:
        return None
    
    # Try different patterns to extract the answer letter
    patterns = [
        r"Answer:\s*([A-J])",             # "Answer: A"
        r"Answer is\s*([A-J])",           # "Answer is A"
        r"answer is\s*([A-J])",           # "answer is A"
        r"The answer is\s*([A-J])",       # "The answer is A"
        r"the answer is\s*([A-J])",       # "the answer is A"
        r"Option\s*([A-J])",              # "Option A"
        r"option\s*([A-J])",              # "option A"
        r"My answer is\s*([A-J])",        # "My answer is A"
        r"(\n|^)([A-J])\.?\s*$",          # "A." or just "A" at end or newline
        r"select option\s*([A-J])",       # "select option A"
        r"I select\s*([A-J])",            # "I select A"
        r"I choose\s*([A-J])",            # "I choose A"
    ]
    
    for pattern in patterns:
        match = re.search(pattern, text)
        if match:
            # Some patterns have the letter in group 1, others in group 2
            return match.group(1) if len(match.groups()) == 1 else match.group(2)
    
    # If no match found, check if there's a single letter at the end
    words = text.strip().split()
    if words and len(words[-1]) == 1 and words[-1].isalpha() and words[-1].upper() in "ABCDEFGHIJ":
        return words[-1].upper()
    
    return None


# At the beginning, before the loop
progress_file = paths.PREDICTIONS / "MedGemma72B_predictions_on_GPT5_progress.csv"

# Check if progress file exists and load it
if os.path.exists(progress_file):
    existing_results = pd.read_csv(progress_file)
    # Extract processed row indices from QA_ID (format: "Merge Q123")
    processed_indices = set()
    for qa_id in existing_results['QA_ID']:
        # Extract number from "Merge Q123" -> 123, then convert to 0-indexed (122)
        idx = int(qa_id.split('Q')[1]) - 1
        processed_indices.add(idx)
    
    results = existing_results.to_dict('records')
    print(f"Found {len(processed_indices)} already processed rows. Resuming...")
else:
    processed_indices = set()
    results = []
    print("Starting from scratch...")


# Loop through the dataset
total_rows = len(df)
print(f"Processing {total_rows} rows...")

for idx, row in df.head(total_rows).iterrows():
    # Skip if already processed
    if idx in processed_indices:
        print(f"Skipping row {idx+1}/{total_rows} (already processed)...")
        continue
    
    print(f"Processing row {idx+1}/{total_rows}...")
    try:
        context_text = row["centaur_question_corr"]
        question = row["question_options"]
        
        # Improved prompt with clearer instructions
        prompt = (
            "You are a clinical reasoning assistant. You will receive a patient case summary "
            "and a multiple-choice question.\n\n"
            f"{context_text}\n\n"
            f"{question}\n\n"
            "Please select the single most appropriate answer. Respond only in the following format:\n\n"
            "Answer: <LETTER>"
        )
    
        # Use chat template format (like your working example)
        messages = [
            {"role": "system", "content": "You are Llama. You are a helpful medical assistant."},
            {"role": "user", "content": prompt}
        ]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        
        # Generate prediction
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=100,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Extract only the generated part (not the input)
        generated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        
        # Decode the response
        raw_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # Extract the answer letter
        extracted_answer = extract_answer_letter(raw_response)
        
        
        # If still no answer found, log more details for debugging
        if extracted_answer is None:
            print(f"⚠️ Could not extract answer from response for row {idx+1}:")
            print(f"Response: {raw_response[:100]}...")
        
        # Create result entry
        qa_id = f"Merge Q{idx + 1}"
        result_entry = {
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": raw_response,
            "Extracted_Answer": extracted_answer
        }
        
        results.append(result_entry)
        processed_indices.add(idx)
        print(f"✅ Processed {qa_id}: Answer = {extracted_answer}")
        
        # Save progress every 10 items (increased frequency for safety)
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results)
            temp_df.to_csv(progress_file, index=False)
            print(f"Saved progress to CSV after {idx+1} items")

    except Exception as e:
        print(f"❌ Error on row {idx}: {str(e)}")
        # Still try to save the entry with error info
        qa_id = f"Merge Q{idx + 1}"
        results.append({
            "QA_ID": qa_id,
            "Origin": row.get("Origin", ""),
            "data_source": row.get("data_source_corr", ""),
            "Raw_Response": f"ERROR: {str(e)}",
            "Extracted_Answer": None
        })
        processed_indices.add(idx)

# Save final results
output_df = pd.DataFrame(results)
output_file = paths.PREDICTIONS / "MedGemma72B_predictions_on_GPT5.csv"
output_df.to_csv(output_file, index=False)
print(f"Saved all predictions to {output_file}")

Columns in dataset:
['Unnamed: 0', 'ID_corr', 'centaur_question_corr', 'answer_corr', 'data_source_corr', 'majority_vote', 'run1_response', 'run2_response', 'run3_response', 'num_responses', 'question_options']
Found 770 already processed rows. Resuming...
Processing 1300 rows...
Skipping row 1/1300 (already processed)...
Skipping row 2/1300 (already processed)...
Skipping row 3/1300 (already processed)...
Skipping row 4/1300 (already processed)...
Skipping row 5/1300 (already processed)...
Skipping row 6/1300 (already processed)...
Skipping row 7/1300 (already processed)...
Skipping row 8/1300 (already processed)...
Skipping row 9/1300 (already processed)...
Skipping row 10/1300 (already processed)...
Skipping row 11/1300 (already processed)...
Skipping row 12/1300 (already processed)...
Skipping row 13/1300 (already processed)...
Skipping row 14/1300 (already processed)...
Skipping row 15/1300 (already processed)...
Skipping row 16/1300 (already processed)...
Skipping row 17/1300 (al

✅ Processed Merge Q771: Answer = B
Processing row 772/1300...
✅ Processed Merge Q772: Answer = B
Processing row 773/1300...
⚠️ Could not extract answer from response for row 773:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the best nex...
✅ Processed Merge Q773: Answer = None
Processing row 774/1300...
⚠️ Could not extract answer from response for row 774:
Response: <unused94>thought
The user wants me to identify the most likely cause of the patient's symptoms base...
✅ Processed Merge Q774: Answer = None
Processing row 775/1300...
✅ Processed Merge Q775: Answer = A
Processing row 776/1300...
⚠️ Could not extract answer from response for row 776:
Response: <unused94>thought
The user wants me to act as a medical assistant and clinical reasoning assistant. ...
✅ Processed Merge Q776: Answer = None
Processing row 777/1300...
⚠️ Could not extract answer from response for row 777:
Response: <unused94>thought
The user wants me to identify

✅ Processed Merge Q837: Answer = A
Processing row 838/1300...
✅ Processed Merge Q838: Answer = A
Processing row 839/1300...
✅ Processed Merge Q839: Answer = B
Processing row 840/1300...
⚠️ Could not extract answer from response for row 840:
Response: <unused94>thought
The user wants me to identify the most likely diagnosis based on the provided clin...
✅ Processed Merge Q840: Answer = None
Saved progress to CSV after 840 items
Processing row 841/1300...
⚠️ Could not extract answer from response for row 841:
Response: <unused94>thought
The user wants me to identify the most likely genetic cause for the patient's cond...
✅ Processed Merge Q841: Answer = None
Processing row 842/1300...
⚠️ Could not extract answer from response for row 842:
Response: <unused94>thought
The user wants me to identify the most appropriate next step in managing a pregnan...
✅ Processed Merge Q842: Answer = None
Processing row 843/1300...
✅ Processed Merge Q843: Answer = H
Processing row 844/1300...
✅ Processed 

✅ Processed Merge Q904: Answer = B
Processing row 905/1300...
✅ Processed Merge Q905: Answer = C
Processing row 906/1300...
✅ Processed Merge Q906: Answer = E
Processing row 907/1300...
✅ Processed Merge Q907: Answer = A
Processing row 908/1300...
✅ Processed Merge Q908: Answer = C
Processing row 909/1300...
⚠️ Could not extract answer from response for row 909:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the most app...
✅ Processed Merge Q909: Answer = None
Processing row 910/1300...
✅ Processed Merge Q910: Answer = B
Saved progress to CSV after 910 items
Processing row 911/1300...
⚠️ Could not extract answer from response for row 911:
Response: <unused94>thought
The user wants me to identify the most likely diagnosis based on the provided pati...
✅ Processed Merge Q911: Answer = None
Processing row 912/1300...
✅ Processed Merge Q912: Answer = D
Processing row 913/1300...
✅ Processed Merge Q913: Answer = C
Processing row 914/1300..

✅ Processed Merge Q980: Answer = B
Saved progress to CSV after 980 items
Processing row 981/1300...
✅ Processed Merge Q981: Answer = D
Processing row 982/1300...
✅ Processed Merge Q982: Answer = A
Processing row 983/1300...
✅ Processed Merge Q983: Answer = C
Processing row 984/1300...
⚠️ Could not extract answer from response for row 984:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the best nex...
✅ Processed Merge Q984: Answer = None
Processing row 985/1300...
✅ Processed Merge Q985: Answer = B
Processing row 986/1300...
✅ Processed Merge Q986: Answer = B
Processing row 987/1300...
✅ Processed Merge Q987: Answer = A
Processing row 988/1300...
✅ Processed Merge Q988: Answer = C
Processing row 989/1300...
✅ Processed Merge Q989: Answer = B
Processing row 990/1300...
✅ Processed Merge Q990: Answer = B
Saved progress to CSV after 990 items
Processing row 991/1300...
✅ Processed Merge Q991: Answer = C
Processing row 992/1300...
✅ Proces

✅ Processed Merge Q1045: Answer = A
Processing row 1046/1300...
⚠️ Could not extract answer from response for row 1046:
Response: <unused94>thought
The user wants me to identify the most likely causal organism for acute bacterial ...
✅ Processed Merge Q1046: Answer = None
Processing row 1047/1300...
⚠️ Could not extract answer from response for row 1047:
Response: <unused94>thought
The user wants me to analyze a patient case and choose the best recommendation reg...
✅ Processed Merge Q1047: Answer = None
Processing row 1048/1300...
⚠️ Could not extract answer from response for row 1048:
Response: <unused94>thought
The user wants me to identify the most likely accumulated substance in the patient...
✅ Processed Merge Q1048: Answer = None
Processing row 1049/1300...
⚠️ Could not extract answer from response for row 1049:
Response: <unused94>thought
The user wants me to analyze a patient case snippet and choose the best multiple-c...
✅ Processed Merge Q1049: Answer = None
Processing row 1

✅ Processed Merge Q1102: Answer = B
Processing row 1103/1300...
✅ Processed Merge Q1103: Answer = B
Processing row 1104/1300...
✅ Processed Merge Q1104: Answer = C
Processing row 1105/1300...
✅ Processed Merge Q1105: Answer = C
Processing row 1106/1300...
⚠️ Could not extract answer from response for row 1106:
Response: <unused94>thought
The user wants me to identify the most likely diagnosis based on the provided pati...
✅ Processed Merge Q1106: Answer = None
Processing row 1107/1300...
✅ Processed Merge Q1107: Answer = D
Processing row 1108/1300...
✅ Processed Merge Q1108: Answer = C
Processing row 1109/1300...
✅ Processed Merge Q1109: Answer = H
Processing row 1110/1300...
✅ Processed Merge Q1110: Answer = C
Saved progress to CSV after 1110 items
Processing row 1111/1300...
✅ Processed Merge Q1111: Answer = A
Processing row 1112/1300...
✅ Processed Merge Q1112: Answer = G
Processing row 1113/1300...
⚠️ Could not extract answer from response for row 1113:
Response: <unused94>thought


✅ Processed Merge Q1167: Answer = C
Processing row 1168/1300...
⚠️ Could not extract answer from response for row 1168:
Response: <unused94>thought
The user wants me to identify the most likely complication for a patient presentin...
✅ Processed Merge Q1168: Answer = None
Processing row 1169/1300...
✅ Processed Merge Q1169: Answer = C
Processing row 1170/1300...
✅ Processed Merge Q1170: Answer = B
Saved progress to CSV after 1170 items
Processing row 1171/1300...
⚠️ Could not extract answer from response for row 1171:
Response: <unused94>thought
The user wants me to identify the most likely underlying cause of the patient's ch...
✅ Processed Merge Q1171: Answer = None
Processing row 1172/1300...
✅ Processed Merge Q1172: Answer = C
Processing row 1173/1300...
⚠️ Could not extract answer from response for row 1173:
Response: <unused94>thought
The user wants me to identify the most likely diagnosis based on the provided clin...
✅ Processed Merge Q1173: Answer = None
Processing row 1174/13

✅ Processed Merge Q1231: Answer = B
Processing row 1232/1300...
✅ Processed Merge Q1232: Answer = B
Processing row 1233/1300...
⚠️ Could not extract answer from response for row 1233:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the most app...
✅ Processed Merge Q1233: Answer = None
Processing row 1234/1300...
✅ Processed Merge Q1234: Answer = F
Processing row 1235/1300...
✅ Processed Merge Q1235: Answer = C
Processing row 1236/1300...
⚠️ Could not extract answer from response for row 1236:
Response: <unused94>thought
The user wants me to act as a clinical reasoning assistant and choose the best nex...
✅ Processed Merge Q1236: Answer = None
Processing row 1237/1300...
✅ Processed Merge Q1237: Answer = H
Processing row 1238/1300...
⚠️ Could not extract answer from response for row 1238:
Response: <unused94>thought
The user wants me to analyze a clinical case and choose the most appropriate manag...
✅ Processed Merge Q1238: Answer = No

✅ Processed Merge Q1300: Answer = C
Saved progress to CSV after 1300 items
Saved all predictions to MedGemma72B_predictions_on_GPT5.csv


In [5]:
import pandas as pd
import numpy as np
from scipy import stats

# Load the data
output_df = pd.read_csv(paths.PREDICTIONS / "MedGemma72B_predictions_on_GPT5.csv")
df = pd.read_csv(paths.DATA / "gpt5-irr-removed-relevancy-combined-dec-12.csv")

# Ensure both dataframes have the same length
assert len(output_df) == len(df), "DataFrames have different lengths!"

# Calculate None/Empty cells in Extracted_Answer - OVERALL
total_empty_cells = output_df['Extracted_Answer'].isna().sum() + (output_df['Extracted_Answer'] == '').sum()
total_cells = len(output_df)
empty_percentage = (total_empty_cells / total_cells) * 100

print("=" * 60)
print("NONE/EMPTY CELL ANALYSIS - OVERALL")
print("=" * 60)
print(f"Total None/Empty cells in 'Extracted_Answer': {total_empty_cells}")
print(f"Total cells: {total_cells}")
print(f"Percentage None/Empty: {empty_percentage:.2f}%")
print("=" * 60)
print()

# Calculate accuracy (assuming both columns contain the same type of answers to compare)
# Method 1: Exact match
output_df['match'] = (output_df['Extracted_Answer'] == df['answer_corr']).astype(int)

# Overall accuracy statistics
accuracy = output_df['match'].mean()
std_dev = output_df['match'].std()
n = len(output_df)
se = std_dev / np.sqrt(n)  # Standard error
ci_95 = stats.t.interval(0.95, n-1, loc=accuracy, scale=se)

print("=" * 60)
print("OVERALL ACCURACY ANALYSIS")
print("=" * 60)
print(f"Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Standard Deviation: {std_dev:.4f}")
print(f"95% Confidence Interval: [{ci_95[0]:.4f}, {ci_95[1]:.4f}]")
print(f"95% CI (percentage): [{ci_95[0]*100:.2f}%, {ci_95[1]*100:.2f}%]")
print(f"Sample Size: {n}")
print("=" * 60)
print()

# Analysis by category (assuming data_source_corr is in one of the dataframes)
# Check which dataframe has data_source_corr
if 'data_source_corr' in output_df.columns:
    analysis_df = output_df.copy()
elif 'data_source_corr' in df.columns:
    analysis_df = output_df.copy()
    analysis_df['data_source_corr'] = df['data_source_corr']
else:
    print("Warning: 'data_source_corr' column not found in either dataframe")
    analysis_df = output_df.copy()

# Category-wise analysis
if 'data_source_corr' in analysis_df.columns:
    # Calculate None/Empty cells by data_source
    empty_by_source = analysis_df.groupby('data_source_corr').apply(
        lambda x: pd.Series({
            'None_Count': x['Extracted_Answer'].isna().sum(),
            'Empty_String_Count': (x['Extracted_Answer'] == '').sum(),
            'Total_Empty': x['Extracted_Answer'].isna().sum() + (x['Extracted_Answer'] == '').sum()
        })
    ).reset_index()
    
    source_totals = analysis_df.groupby('data_source_corr').size().reset_index(name='Total_Count')
    empty_summary = empty_by_source.merge(source_totals, on='data_source_corr')
    empty_summary['Empty_Percentage'] = (empty_summary['Total_Empty'] / empty_summary['Total_Count']) * 100
    
    print("NONE/EMPTY CELLS BY DATA SOURCE")
    print("=" * 60)
    print(empty_summary.to_string(index=False))
    print("=" * 60)
    print()
    
    category_stats = analysis_df.groupby('data_source_corr')['match'].agg([
        ('Count', 'count'),
        ('Mean_Accuracy', 'mean'),
        ('Std_Dev', 'std'),
        ('SE', lambda x: x.std() / np.sqrt(len(x)))
    ]).reset_index()
    
    # Calculate 95% CI for each category
    ci_lower = []
    ci_upper = []
    
    for idx, row in category_stats.iterrows():
        n_cat = row['Count']
        mean_cat = row['Mean_Accuracy']
        se_cat = row['SE']
        
        if n_cat > 1:
            ci = stats.t.interval(0.95, n_cat-1, loc=mean_cat, scale=se_cat)
            ci_lower.append(ci[0])
            ci_upper.append(ci[1])
        else:
            ci_lower.append(np.nan)
            ci_upper.append(np.nan)
    
    category_stats['CI_95_Lower'] = ci_lower
    category_stats['CI_95_Upper'] = ci_upper
    
    # Format percentages
    category_stats['Mean_Accuracy_%'] = category_stats['Mean_Accuracy'] * 100
    category_stats['Std_Dev_%'] = category_stats['Std_Dev'] * 100
    category_stats['CI_95_Lower_%'] = category_stats['CI_95_Lower'] * 100
    category_stats['CI_95_Upper_%'] = category_stats['CI_95_Upper'] * 100
    
    print("CATEGORY-WISE ACCURACY ANALYSIS")
    print("=" * 60)
    print(category_stats.to_string(index=False))
    print("=" * 60)
    print()
    
# Create summary statistics table
summary_table = pd.DataFrame({
    'Metric': ['Overall Accuracy', 'Standard Deviation', '95% CI Lower', '95% CI Upper', 'Sample Size'],
    'Value': [f"{accuracy:.4f} ({accuracy*100:.2f}%)", 
              f"{std_dev:.4f}", 
              f"{ci_95[0]:.4f} ({ci_95[0]*100:.2f}%)", 
              f"{ci_95[1]:.4f} ({ci_95[1]*100:.2f}%)", 
              n]
})

print("\nSUMMARY TABLE")
print("=" * 60)
print(summary_table.to_string(index=False))
print("=" * 60)

NONE/EMPTY CELL ANALYSIS - OVERALL
Total None/Empty cells in 'Extracted_Answer': 458
Total cells: 1300
Percentage None/Empty: 35.23%

OVERALL ACCURACY ANALYSIS
Accuracy: 0.4038 (40.38%)
Standard Deviation: 0.4909
95% Confidence Interval: [0.3771, 0.4306]
95% CI (percentage): [37.71%, 43.06%]
Sample Size: 1300

NONE/EMPTY CELLS BY DATA SOURCE
data_source_corr  None_Count  Empty_String_Count  Total_Empty  Total_Count  Empty_Percentage
            jama         208                   0          208          582         35.738832
      medbullets          41                   0           41          207         19.806763
        medxpert         144                   0          144          318         45.283019
            mmlu          65                   0           65          193         33.678756

CATEGORY-WISE ACCURACY ANALYSIS
data_source_corr  Count  Mean_Accuracy  Std_Dev       SE  CI_95_Lower  CI_95_Upper  Mean_Accuracy_%  Std_Dev_%  CI_95_Lower_%  CI_95_Upper_%
            jama 

/tmp/ipykernel_1367566/4253476119.py:62: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  empty_by_source = analysis_df.groupby('data_source_corr').apply(
